# Part 4 — NVIDIA RAPIDS: CPU vs GPU Comparison

This notebook compares familiar CPU data-science tools with their GPU-accelerated RAPIDS equivalents.

The important comparisons are:

1. NumPy vs CuPy
2. pandas vs cuDF
3. scikit-learn K-Means vs GPU-accelerated K-Means
4. Large workloads vs small workloads

**Before running:** In Colab choose **Runtime → Change runtime type → T4 GPU**.

The benchmark sizes are intentionally moderate so the complete notebook can run reliably in Colab while still showing the same CPU-vs-GPU concepts.


## 1. Setup and Verify the GPU

### What we saw before
- This is the starting point of the RAPIDS comparison.
- GPU benchmarks are meaningful only when a supported NVIDIA GPU is actually available.

### How this cell is different
- This cell prepares the CPU and GPU libraries instead of running a benchmark.
- It avoids reinstalling cuDF because cuDF is already available on Colab GPU runtimes.
- cuML is installed only if it is missing.

### What this cell does
- Runs `nvidia-smi` to verify the GPU.
- Imports NumPy and pandas for CPU work.
- Imports CuPy and cuDF for GPU work.
- Installs only `cuML` if needed.
- Defines a small benchmark helper with GPU synchronization.

### What we achieve after running it
- CPU and GPU libraries are ready.
- We can confirm the actual GPU model and RAPIDS versions.

### Remember
**NumPy → CPU arrays**

**CuPy → GPU arrays**

**pandas → CPU DataFrames**

**cuDF → GPU DataFrames**

**scikit-learn / cuML → CPU / GPU machine learning**


In [ ]:
import sys
import time
import subprocess
import importlib.util
import numpy as np
import pandas as pd

# Verify that Colab really has a GPU.
gpu_check = subprocess.run(
    ["nvidia-smi"],
    capture_output=True,
    text=True
)

if gpu_check.returncode != 0:
    raise RuntimeError(
        "No NVIDIA GPU detected. In Colab choose "
        "Runtime -> Change runtime type -> T4 GPU, then reconnect."
    )

print(gpu_check.stdout.splitlines()[0])
print("GPU detected successfully.")

# CuPy and cuDF are available on Colab GPU runtimes.
import cupy as cp
import cudf

# Install only cuML if it is not already present.
if importlib.util.find_spec("cuml") is None:
    cuda_major = cp.cuda.runtime.runtimeGetVersion() // 1000
    if cuda_major not in (12, 13):
        raise RuntimeError(f"Unsupported CUDA major version: {cuda_major}")

    pkg = f"cuml-cu{cuda_major}"
    print(f"{pkg} not found. Installing only cuML...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        pkg,
        "--extra-index-url=https://pypi.nvidia.com"
    ])

import cuml

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

def sync_gpu():
    cp.cuda.Device().synchronize()

def bench(fn, repeat=2, gpu=False):
    # Warm-up
    fn()
    if gpu:
        sync_gpu()

    times = []
    for _ in range(repeat):
        start = time.perf_counter()
        fn()
        if gpu:
            sync_gpu()
        times.append(time.perf_counter() - start)

    return min(times)

print("cuDF version:", cudf.__version__)
print("cuML version:", cuml.__version__)
print("CUDA runtime:", cp.cuda.runtime.runtimeGetVersion())


## 2. NumPy vs CuPy — Same Array Operation, CPU vs GPU

### What we saw before
- The previous cell only confirmed that CPU and GPU libraries are available.
- Now we perform the first real benchmark.

### How this cell is different
- We use the same large array operation on two different processors.
- NumPy performs the sort on the CPU.
- CuPy performs the sort on the GPU.

### What this cell does
- Creates one million random `float32` values in host memory.
- `np.sort(x_cpu)` sorts them on the CPU.
- `cp.asarray(x_cpu)` copies the data from CPU memory to GPU memory.
- `cp.sort(x_gpu)` performs the same sort on the GPU.
- We measure both the warm GPU computation time and the host-to-device copy time.
- We also verify that CPU and GPU return the same result.

### What we achieve after running it
- We get a measured CPU time and GPU time.
- We also see an important GPU lesson: transferring data to the GPU has a cost.

### Remember
A GPU benchmark has more than one number:

**data-transfer time + first-call overhead + warm computation time**

Do not claim a GPU speed-up while ignoring the cost of moving the data.


In [ ]:
N = 1_000_000

x_cpu = rng.random(N, dtype=np.float32)

cpu_sort_time = bench(
    lambda: np.sort(x_cpu),
    repeat=2,
    gpu=False
)

# Copy CPU array to GPU
start = time.perf_counter()
x_gpu = cp.asarray(x_cpu)
sync_gpu()
copy_time = time.perf_counter() - start

gpu_sort_time = bench(
    lambda: cp.sort(x_gpu),
    repeat=2,
    gpu=True
)

cpu_result = np.sort(x_cpu)
gpu_result = cp.asnumpy(cp.sort(x_gpu))

print(f"Array size: {N:,} float32 values")
print(f"CPU NumPy sort:       {cpu_sort_time:.4f} s")
print(f"CPU -> GPU copy:      {copy_time:.4f} s")
print(f"GPU CuPy sort:        {gpu_sort_time:.4f} s")
print(f"Warm speed-up:        {cpu_sort_time / gpu_sort_time:.2f}x")
print(
    f"Speed-up incl. copy:  "
    f"{cpu_sort_time / (copy_time + gpu_sort_time):.2f}x"
)
print("Results identical:", np.allclose(cpu_result, gpu_result))


## 3. pandas vs cuDF — DataFrame GroupBy

### What we saw before
- NumPy and CuPy compared the same array operation.
- We also saw that copying data to GPU memory has a cost.

### How this cell is different
- Now we compare table/dataframe processing.
- pandas runs the groupby on the CPU.
- cuDF performs a very similar DataFrame operation on the GPU.

### What this cell does
- Creates a large transaction table with two million rows.
- The columns represent customer ID, category, and amount.
- pandas computes average amount per customer using `groupby`.
- `cudf.from_pandas(df)` transfers the table to GPU memory.
- cuDF performs the same `groupby().mean()` operation.
- We compare CPU time, GPU time, and transfer time.

### What we achieve after running it
- We see that the pandas programming style can often be transferred directly to cuDF.
- We can measure whether the larger parallel workload benefits from the GPU.

### Remember
The API is intentionally familiar:

```python
pandas_df.groupby(...).mean()
cudf_df.groupby(...).mean()
```

The major difference is **where the data and computation live**.


In [ ]:
N_ROWS = 500_000

df_cpu = pd.DataFrame({
    "customer_id": rng.integers(0, 25_000, N_ROWS),
    "category": rng.integers(0, 20, N_ROWS),
    "amount": rng.uniform(5, 500, N_ROWS).astype("float32")
})

cpu_groupby_time = bench(
    lambda: df_cpu.groupby("customer_id")["amount"].mean(),
    repeat=2
)

start = time.perf_counter()
df_gpu = cudf.from_pandas(df_cpu)
sync_gpu()
df_copy_time = time.perf_counter() - start

gpu_groupby_time = bench(
    lambda: df_gpu.groupby("customer_id")["amount"].mean(),
    repeat=2,
    gpu=True
)

cpu_grouped = (
    df_cpu.groupby("customer_id")["amount"]
    .mean()
    .sort_index()
)

gpu_grouped = (
    df_gpu.groupby("customer_id")["amount"]
    .mean()
    .sort_index()
    .to_pandas()
)

print(f"Rows: {N_ROWS:,}")
print(f"pandas CPU groupby:   {cpu_groupby_time:.4f} s")
print(f"pandas -> cuDF copy:  {df_copy_time:.4f} s")
print(f"cuDF GPU groupby:     {gpu_groupby_time:.4f} s")
print(f"Warm speed-up:        {cpu_groupby_time / gpu_groupby_time:.2f}x")
print(
    f"Speed-up incl. copy:  "
    f"{cpu_groupby_time / (df_copy_time + gpu_groupby_time):.2f}x"
)
print(
    "Results close:",
    np.allclose(cpu_grouped.values, gpu_grouped.values, rtol=1e-5)
)


## 4. scikit-learn K-Means vs cuML K-Means

### What we saw before
- NumPy/CuPy compared array operations.
- pandas/cuDF compared DataFrame operations.
- Both showed that moving data to the GPU has a cost.

### How this cell is different
- Now we compare an actual machine-learning algorithm.
- scikit-learn runs K-Means on the CPU.
- cuML runs the equivalent K-Means algorithm on the GPU.

### What this cell does
- Creates the same numeric dataset for both implementations.
- Trains `sklearn.cluster.KMeans` on the CPU.
- Transfers the same data to GPU memory.
- Trains `cuml.cluster.KMeans` on the GPU.
- Compares execution time and inertia.

### What we achieve after running it
- We see the machine-learning side of RAPIDS acceleration without a large subprocess or repeated model search.
- The underlying K-Means task remains the same on CPU and GPU.

### Remember
cuML follows a scikit-learn-like API:

```python
model = KMeans(...)
model.fit(X)
```

The major difference is the execution hardware.

RAPIDS also provides `cuml.accel` for supported zero-code-change acceleration, but direct cuML is used here to keep this benchmark reliable and fast.


In [ ]:
from sklearn.cluster import KMeans as SklearnKMeans
from cuml.cluster import KMeans as CuMLKMeans

X_cpu = rng.normal(
    size=(30_000, 10)
).astype("float32")

# CPU K-Means
cpu_model = SklearnKMeans(
    n_clusters=8,
    n_init=1,
    random_state=RANDOM_STATE
)

start = time.perf_counter()
cpu_model.fit(X_cpu)
cpu_kmeans_time = time.perf_counter() - start

# Transfer data to the GPU once
start = time.perf_counter()
X_gpu = cp.asarray(X_cpu)
sync_gpu()
transfer_time = time.perf_counter() - start

# GPU K-Means
gpu_model = CuMLKMeans(
    n_clusters=8,
    n_init=1,
    random_state=RANDOM_STATE
)

# Warm-up on a subset
gpu_model.fit(X_gpu[:3000])
sync_gpu()

start = time.perf_counter()
gpu_model.fit(X_gpu)
sync_gpu()
gpu_kmeans_time = time.perf_counter() - start

comparison = pd.DataFrame([
    {
        "Run": "scikit-learn KMeans (CPU)",
        "Seconds": cpu_kmeans_time,
        "Inertia": float(cpu_model.inertia_)
    },
    {
        "Run": "cuML KMeans (GPU)",
        "Seconds": gpu_kmeans_time,
        "Inertia": float(gpu_model.inertia_)
    }
])

display(comparison.round(4))

print("CPU -> GPU transfer:", round(transfer_time, 4), "s")
print(
    "Warm compute speed-up:",
    round(cpu_kmeans_time / gpu_kmeans_time, 2),
    "x"
)


## 5. Small Data — Why the CPU Can Still Win

### What we saw before
- Large array, DataFrame, and ML workloads can benefit from the GPU.
- But every GPU operation has fixed overhead such as kernel launch and memory transfer.

### How this cell is different
- We intentionally make the dataset very small.
- The purpose is to show that GPU is not automatically faster.

### What this cell does
- Creates only 1,500 rows.
- Runs the same `groupby` in pandas and cuDF.
- Measures both times after warm-up.

### What we achieve after running it
- On small data, pandas may be faster or very competitive.
- This explains why benchmarking must match the real workload.

### Remember
A useful mental model is:

**CPU = sports car**

- low startup overhead
- good for small or sequential work

**GPU = freight train**

- higher startup cost
- excellent when there is enough parallel work to fill it

So the correct question is not:

**"Is GPU faster?"**

It is:

**"Is this workload large and parallel enough for GPU acceleration to pay off?"**


In [ ]:
small_cpu = pd.DataFrame({
    "group": rng.integers(0, 10, 1500),
    "value": rng.random(1500).astype("float32")
})

small_gpu = cudf.from_pandas(small_cpu)

cpu_small_time = bench(
    lambda: small_cpu.groupby("group")["value"].mean(),
    repeat=3
)

gpu_small_time = bench(
    lambda: small_gpu.groupby("group")["value"].mean(),
    repeat=3,
    gpu=True
)

print(f"CPU pandas: {cpu_small_time * 1e6:.0f} microseconds")
print(f"GPU cuDF:   {gpu_small_time * 1e6:.0f} microseconds")

winner = "CPU pandas" if cpu_small_time < gpu_small_time else "GPU cuDF"
print("Faster on this small example:", winner)


# Final Revision Takeaways

### What we saw through the notebook

**1. NumPy vs CuPy**

Same kind of numerical operation, different processor.

↓

**2. pandas vs cuDF**

Similar DataFrame API, but cuDF stores and processes data on the GPU.

↓

**3. scikit-learn vs `cuml.accel`**

Supported sklearn estimators can sometimes move to the GPU without changing the estimator code.

↓

**4. Large data vs small data**

GPU acceleration has overhead, so workload size matters.

---

## Core RAPIDS map

| CPU ecosystem | GPU/RAPIDS equivalent |
|---|---|
| NumPy | CuPy |
| pandas | cuDF |
| scikit-learn | cuML |
| sklearn code | `cuml.accel` for supported estimators |

---

## Most important benchmarking lesson

A fair GPU comparison should consider:

- CPU execution time
- GPU warm execution time
- first-call/JIT overhead
- CPU-to-GPU transfer time
- dataset size
- whether results are equivalent

---

## Main takeaway

**RAPIDS is most valuable when the dataset is large and the operation can be parallelized.**

GPU acceleration is not a replacement for benchmarking.

For small workloads, the CPU can still be the better choice.
